In [1]:
# ============================================================
# 도로 파손 인식 함수 (통합 버전)
# 담당: 민서 (VLM 연동) → 민지 (웹 통합용으로 전달)
#
# 수정 사항:
#   - 최종 선정된 메인 모델 'HyperCLOVA X (HCX-005)' 기반으로 함수 전면 개편
#     (균열·포트홀 정확도 격차가 적어 균형성이 뛰어남: 71.8% / 76.4%)
#   - 보조 모델(Gemini)은 필요 시 비교용으로 쓸 수 있도록 하단에 추가 배치
# ============================================================

from openai import OpenAI
import base64
import re
import os

def analyze_road_damage_clova(image_path, api_key):
    """
    도로 사진을 넣으면 파손 유형과 심각도를 반환하는 메인 함수
    (최종 선정 메인 모델: HyperCLOVA X / HCX-005)

    입력:
        image_path (str): 분석할 사진 파일 경로
        api_key (str): CLOVA Studio API 키

    출력:
        dict 형태: {"유형": "균열" 또는 "포트홀", "심각도": "경미"/"보통"/"심각"/"긴급"}
    """
    client = OpenAI(
        api_key=api_key,
        base_url="https://clovastudio.stream.ntruss.com/v1/openai"
    )

    PROMPT = """이 사진 속 도로 파손을 분석해줘. 아래 형식으로만 정확히 답해줘 (다른 말 붙이지 말고):

유형: 균열 또는 포트홀 (둘 중 하나만)

심각도: 아래 기준을 참고해서 판단해줘
- 경미: 표면에 실금 수준이거나 아주 작은 파임, 즉각적인 위험 없음
- 보통: 육안으로 뚜렷이 보이는 파손, 주의가 필요한 수준
- 심각: 파손 범위가 넓거나 깊어 통행에 지장을 줄 수 있는 수준
- 긴급: 깊은 구멍, 넓은 함몰, 도로 일부 붕괴 등 사고 위험이 임박한 수준

반드시 아래 형식 그대로만 답하고, 기준 설명은 다시 쓰지 마:
유형: [답]
심각도: [답]"""

    with open(image_path, "rb") as f:
        base64_image = base64.b64encode(f.read()).decode("utf-8")

    response = client.chat.completions.create(
        model="HCX-005",
        max_tokens=100,
        messages=[{
            "role": "user",
            "content": [
                {"type": "text", "text": PROMPT},
                {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}}
            ]
        }]
    )

    answer = response.choices[0].message.content
    type_match = re.search(r"유형[:\s]*(균열|포트홀|해당 없음)", answer)
    severity_match = re.search(r"심각도[:\s]*(경미|보통|심각|긴급|해당 없음)", answer)

    return {
        "유형": type_match.group(1) if type_match else "인식 실패",
        "심각도": severity_match.group(1) if severity_match else "인식 실패"
    }


# ============================================================
# [참고] 보조/비교 모델용 Gemini 함수
# ============================================================
from google import genai
from google.genai import types

def analyze_road_damage_gemini(image_path, api_key=None):
    """
    보조 비교 모델 (Gemini Flash Lite) 기반 인식 함수
    """
    if api_key:
        os.environ["GEMINI_API_KEY"] = api_key

    client = genai.Client()

    PROMPT = """이 사진 속 도로 파손을 분석해줘. 아래 형식으로만 정확히 답해줘 (다른 말 붙이지 말고):
유형: 균열 또는 포트홀 (둘 중 하나만)
심각도: 경미/보통/심각/긴급 중 택1"""

    with open(image_path, "rb") as f:
        image_bytes = f.read()

    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=[
            types.Part.from_bytes(
                data=image_bytes,
                mime_type='image/jpeg',
            ),
            PROMPT
        ]
    )

    answer = response.text
    type_match = re.search(r"유형[:\s]*(균열|포트홀|해당 없음)", answer)
    severity_match = re.search(r"심각도[:\s]*(경미|보통|심각|긴급|해당 없음)", answer)

    return {
        "유형": type_match.group(1) if type_match else "인식 실패",
        "심각도": severity_match.group(1) if severity_match else "인식 실패"
    }


# ============================================================
# 웹 화면(Streamlit)에서 사용할 때 참고 예시
# ============================================================
#
# import streamlit as st
#
# uploaded_file = st.file_uploader("도로 사진을 업로드하세요", type=["jpg", "jpeg", "png"])
#
# if uploaded_file is not None:
#     with open("temp.jpg", "wb") as f:
#         f.write(uploaded_file.getbuffer())
#
#     # 🌟 메인 모델(HyperCLOVA X) 함수 호출
#     result = analyze_road_damage_clova("temp.jpg", api_key="본인의_CLOVA_API_키")
#     st.write(f"유형: {result['유형']}")
#     st.write(f"심각도: {result['심각도']}")
#
# ============================================================